# Continuation Method for Pareto Exploration for the problems below

# Problems (3 Objectives)

# Defining Mathematical objectives
Given $C^1, C^2, C^3 \in \R^n$

* # 1

$min_{x\in\R^n}
\begin{bmatrix}
\frac{1}{2}\|x-C^1\|_2^2 \\
\frac{1}{2}\|x-C^2\|_2^2 \\
\frac{1}{2}\|x-C^3\|_2^2 \\
\end{bmatrix}
$

* # 2

$
\min_{x \in \mathbb{R}^n} F(x) = \min_{x \in \mathbb{R}^n} \left\{ \frac{1}{2}(x-C^i)^T Q^i (x-C^i) \right\}_{i=1}^3
$

In [1]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.sparse.linalg import LinearOperator, minres
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from qpsolvers import solve_qp
from source_codes.method import *
np.set_printoptions(precision=8)
np.random.seed(24) #42
#np.set_printoptions(precision=4)

#*********** problem 1 ***********# Uncomment the desired variant
#from source_codes.xC_problem import x_c_problem

#*********** problem 2 ***********# Uncomment the desired variant
from source_codes.xCQ_problem import x_c_problem




In [2]:
#**********Setting up the problem**********#

problem = x_c_problem()
n, m = problem.n, problem.m # n: dimension of x, m: number of objectives
s= 0.1                     # Step size # you can vary it from 0.01 to 0.1 to see how results improve with this parameter.
maxiter = 2         # Maximum allowable number of iterations in MINRES. You can vary it from 1 to 1000 to see how results improve with this parameter.



# Generate the initial Pareto optimal point.
x0 =  problem.sample_pareto_set(20) #np.array([0.24537, 0.89436, 0.15772])  #
f0 = problem.f(x0)
g0 = problem.grad(x0)
print("Initial point: ", x0)
print("Initial function values: ", f0)
print("Initial gradients: ", g0)

Initial point:  [0.17661239 0.9838796  0.43268055]
Initial function values:  [0.34951935 0.21182564 0.64841309]
Initial gradients:  [[-0.1346719   0.40750827  0.43268055]
 [ 0.00354017 -0.5161204   0.3620356 ]
 [ 0.17661239 -0.11324402 -0.87860373]]


Next, we solve $\alpha$ from Equation (3) in the paper:


### Alpha computation

In [3]:
def compute_alpha(g,m=3):
    alpha_ = cp.Variable(m)
    objective = cp.Minimize(cp.sum_squares(alpha_.T @ g))
    constraints = [alpha_ >= 0, cp.sum(alpha_) == 1]
    alpha_prob = cp.Problem(objective, constraints)
    optimal_loss = alpha_prob.solve()
    alpha_ = ndarray(alpha_.value).ravel()
    return alpha_
alpha = compute_alpha(g0)
print("Initial alphas: ", alpha)

Initial alphas:  [0.42105263 0.26315789 0.31578947]


### Corrector Step

In [4]:
## Using MGDA with line search to find a Pareto optimal solution x* and as a corrector step.

gamma = 0.9         # The decay factor in line search.
num_exp = 10        # How many times you want to repeat the experiment with a different seed.
# Hyperparameters for line search.
ths = 1e-5          # For detecting convergence. Smaller value takes longer to converge.
eta_init =  1        # Initial step size in line search.
c1 = 0              # For strong Wolfe condition. Cannot be negative. Typically 1e-4. Larger values requires more iterations in line search.

def line_search(x, f, grad, d, eta, c1):
    x = ndarray(x).ravel()
    assert x.size == n
    f = ndarray(f).ravel()
    assert f.size == m
    grad = ndarray(grad)
    assert grad.shape == (m, n)
    d = ndarray(d).ravel()
    assert d.size == n
    while True:
        x_new = x + eta * d
        f_new = problem.f(x_new)
        if np.all([fi_new <= fi + c1 * gradi.dot(d) * eta for fi, gradi, fi_new in zip(f, grad, f_new)]):
            return eta
        eta *= gamma

def mgda_optimize(x):
    x = ndarray(x).ravel()
    assert x.size == n
    x_iter = np.copy(x)
    while True:
        g_iter = problem.grad(x_iter)
        f_iter = problem.f(x_iter)
        alpha_iter = compute_alpha(g_iter)
        # Negative sign here because d must be a *descent* direction.
        d = -ndarray(alpha_iter.T @ g_iter).ravel()
        # Make sure they are indeed descent.
        for gi in g_iter:
            assert gi.dot(d) <= 0 or np.isclose(gi.dot(d), 0)
        # Termination condition 1: gradient is too small.
        if np.linalg.norm(d) < ths:
            return x_iter
        eta = line_search(x_iter, f_iter, g_iter, d, eta_init, c1)
        x_iter += eta * d
        # Termination condition 2: change is too little. Effectively, this means eta is too small.
        if eta * np.linalg.norm(d) < ths:
            print("change too small, increase step size")
            return x_iter 
        
#mgda_optimize(np.array([2.0,1.5,0.0]))

### Predictor Step
Computing projected directions influenced by preference vector computed via MINRES

In [ ]:
def minres_solver(problem, x0, alpha,b,  reg = 0.0):
    n = len(b)
    x, _ = minres(LinearOperator((n, n), matvec=lambda v: problem.hvp(x0, alpha, v,reg = reg), rmatvec=lambda v: problem.hvp(x0, alpha, v,reg=reg)), b, maxiter=maxiter) #, rmatvec=H_op
    #print(f"MINRES info: {_}")
    return np.array(x)



def beta_basis(m):
    """Construct n x k matrix with row-sum 0 and full row rank (works if n <= k-1)."""
    k = m-1
    if k > m:
        raise ValueError("Impossible: need n <= k-1 for full row rank with row-sum 0.")
    B = np.zeros((k, m), dtype=float)
    for i in range(k):
        B[i, i] = 1.0
        B[i, i+1] = -1.0
    return B

def generate_span_vi(problem, x0, alpha, reg, n_obj=3):
    """
    Generates two orthogonal vectors v1, v2 in the span of beta1, beta2.
    Returns: (v1, v2,...) as a tuple of numpy arrays.
    """

    vi = []
    beta = beta_basis(n_obj)
    for b in beta:
        rhs = problem.grad(x0).T @ b  # shape: (n,)
        v = minres_solver(problem, x0, alpha, rhs, reg= reg)
        v = v if np.linalg.norm(v) == 0 else v/np.linalg.norm(v) 
        vi.append(v)
    # Compute reduced QR of v.T (shape 3 x 2 -> Q: 3x2, R: 2x2)
    vi = np.array(vi)  # shape k x m
    Q, R = np.linalg.qr(vi.T, mode='reduced')

    # Make R have positive diagonal (canonical QR)
    diag_sign = np.sign(np.diag(R))
    diag_sign[diag_sign == 0] = 1.0
    Q = Q * diag_sign  # broadcast to columns

    # Orthonormal rows are the transpose of Q
    v_orth = Q.T  # shape k x m
    return v_orth

def proj_general(V, d):
    V = np.asarray(V, float).T          # (n,k)
    d = np.asarray(d, float).ravel()  # (n,)
    M = V.T @ V              # (k,k)
    #print(V.shape, d.shape, M.shape)     
    if np.allclose(M, np.eye(M.shape[0])):
        return V @ (V.T @ d)  # (n,)  # V.T @ d    (k,)
    return V @ np.linalg.solve(M, V.T @ d)

In [14]:
def compute_descent_direction(problem, x0, alpha, grads, pref, solvers = "minres", reg =0):
    """
    V: (k, n) array of v_j vectors
    grad_f: (m, n) array of gradients ∇f_i(x)
    P: (m,) array of weights P_i
    Returns: optimal descent direction d ∈ ℝ^n
    """
    selected_grads = []
    selected_prefs = []
    for i in range(len(pref)):
        if pref[i] < 0:
            selected_grads.append(grads[i])
            selected_prefs.append(pref[i])
            
    if len(selected_grads) >= 1:
        G = np.vstack(selected_grads).T  # (n, k) matrix of selected gradients
        #assert G.shape[1] == len(selected_grads), "Number of selected gradients must match the number of columns in G."     

        m = G.shape[1]  # Number of selected gradients
        n = G.shape[0]  # Number of features
        
        P = np.matmul(G.T, G)
        Theta = np.zeros((m, m))         # columns will be θ^i
        D = np.zeros((n, m))             # columns will be d_i

        for i in range(m):
            q = G.T @ G[:, i] #P[:, i] not P[i, :]
            theta_i = solve_qp(P, q, G = None, h = None, A =  None, b =  None, lb = np.zeros((m)), ub = None, solver="cvxopt")
            Theta[:, i] = theta_i
            D[:, i] = G[:, i] +  G @ theta_i

    
        d =  D @ np.array(selected_prefs) 
        vi = generate_span_vi(problem, x0, alpha, reg,grads.shape[0])  # Generate orthonormal basis in the span of beta vectors
        proj_d = proj_general(vi, d)

        #print("d", d)
        #print("proj_d", proj_d)
        return proj_d
    
    else:
        d = grads.T @ pref  # If no gradients are selected, use the full gradient
        # Solve for v
        vi = generate_span_vi(problem, x0, alpha, reg,grads.shape[0])  # Generate orthonormal basis in the span of beta vectors
        proj_d = proj_general(vi, d)
        v = proj_d
        return v



In [11]:
# Example preference vectors for 3 objectives. You can change them as you wish.

"""Ps = [np.array([-0.1, -0.9, 0]),np.array([-0.2,-0.8, 0]),np.array([-0.3, -0.7, 0]),np.array([-0.4, -0.6, 0]),
        np.array([-0.5, -0.5, 0]),np.array([-0.6, -0.4, 0]),np.array([-0.7, -0.3, 0]),np.array([-0.8, -0.2, 0]),np.array([-0.9, -0.1, 0])]
"""


"""Ps = [np.array([-0.1,0,-0.9]),np.array([-0.2,0,-0.8]),np.array([-0.3,0, -0.7]),np.array([-0.4,0, -0.6]),np.array([-0.5,0, -0.5]),
        np.array([-0.6,0, -0.4]),np.array([-0.7,0, -0.3]),np.array([-0.8,0, -0.2]),np.array([-0.9,0, -0.1])]
"""


"""Ps = [np.array([0,-0.1, -0.9]),np.array([0,-0.2,-0.8]),np.array([0,-0.3, -0.7]),np.array([0,-0.4, -0.6]),np.array([0,-0.5, -0.5]),
        np.array([0,-0.6, -0.4]),np.array([0,-0.7, -0.3]),np.array([0,-0.8, -0.2]),np.array([0,-0.9, -0.1])]
"""

Ps = [np.array([-0.1, -0.9, 0]),np.array([-0.2,-0.8, 0]),np.array([-0.3, -0.7, 0]),np.array([-0.4,-0.6, 0]),np.array([-0.5,-0.5, 0]),
      np.array([-0.6, -0.4, 0]),np.array([-0.7, -0.3, 0]),np.array([-0.8, -0.2, 0]),np.array([-0.9, -0.1, 0])]

In [15]:
%matplotlib tk 

# Comment this line if you want to visualize the plots in the notebook itself.
#%matplotlib inline # Uncomment this line if you want to visualize the plots in the notebook itself.


norms = np.linalg.norm(g0, axis=1)
g_normalized = g0 / norms[:, np.newaxis]


points = []
for i, pref in enumerate(Ps):
    d= compute_descent_direction(problem, x0, alpha, g_normalized, pref, solvers = "minres", reg =0.1) 
    points.append((d,  'tab:orange', 'MINRES' if i == 0 else None, -s, s, 2))
points.append((g0[0], 'tab:blue', r'$\nabla f_1(x^*)$', 0, s, 4))
points.append((g0[1], 'tab:green', r'$\nabla f_2(x^*)$', 0, s, 4))
points.append((g0[2], 'black', r'$\nabla f_3(x^*)$', 0, s, 4))


# Visualization in decision space and objective space

fig_ps = plt.figure(figsize=(10, 10))
ax_ps = fig_ps.add_subplot(111, projection='3d', proj_type='ortho')
#problem.plot_pareto_set(ax_ps)
ax_ps.scatter(x0[0], x0[1], x0[2], c='tab:red', s=50)

for idx, (v, color, label,_,_,_) in enumerate(points):
    #draw_arrow_3d(ax_ps, x0 + v / np.linalg.norm(v), x0, color)
    
    if idx == len(points)-1 or idx ==len(points)-2 or idx ==len(points)-3:
        new_x = x0 - s *v / (np.linalg.norm(v))
        ax_ps.plot([x0[0], new_x[0]], [x0[1], new_x[1]], [x0[2],new_x[2]],"-", c=color,label = label,linewidth=2)
    else:
        new_x = x0 + s *v / (np.linalg.norm(v))
        ax_ps.plot([x0[0], new_x[0]], [x0[1], new_x[1]], [x0[2],new_x[2]],"-", c=color,label = label,linewidth=2)
        corr= mgda_optimize(new_x) #corrector_step(new_x, L_vec_func = problem.f, G_mat_func = problem.grad, sigma = 0.01, lr=s,h_mode="exact", h_kwargs=None) #mgda_optimize(new_x)
        ax_ps.plot([new_x[0], corr[0]], [new_x[1], corr[1]], [new_x[2],corr[2]],"-o", c="blue",linewidth=2)
        if idx ==0:
            ax_ps.plot([new_x[0], corr[0]], [new_x[1], corr[1]], [new_x[2],corr[2]],"-o", c="blue",label = "corrector",linewidth=2)


ax_ps.set_xlabel(r"$𝓍_1$",fontsize = 20)
ax_ps.set_ylabel(r'$𝓍_2$',fontsize = 20)
ax_ps.set_zlabel(r'$𝓍_3$', fontsize = 20)
ax_ps.set_title(r'$Decision \  space \ (\eta = 0.5)$', fontsize = 14)
ax_ps.tick_params(axis='x', labelsize=14)
ax_ps.tick_params(axis='y', labelsize=14)
ax_ps.tick_params(axis='z', labelsize=14)
ax_ps.legend(fontsize = 14)
#plt.tight_layout()
plt.show()




fig_pf = plt.figure(figsize=(10, 10))
ax_pf = fig_pf.add_subplot(111, projection='3d')
ax_pf.scatter(f0[0], f0[1],f0[2], c='tab:red', label='Pareto optimal $f(x^*)$', s=50)
problem.plot_pareto_front(ax_pf, label='Pareto front')
print("f0", f0)
for idx, (v, color, label, s0, s1, _) in enumerate(points):
    
    if idx == len(points)-1 or idx ==len(points)-2 or idx ==len(points)-3:
        fi = problem.f(x0 - s * v/ np.linalg.norm(v))
        print(fi)
        ax_pf.plot([f0[0], fi[0]], [f0[1], fi[1]], [f0[2],fi[2]],"-", c=color,label = label,linewidth=2)

    else:
        fi = problem.f(x0 + s * v/ np.linalg.norm(v) )
        #print(fi, "h")
        ax_pf.plot([f0[0], fi[0]], [f0[1], fi[1]], [f0[2],fi[2]],"-", c=color,label = label,linewidth=2)
        corr= problem.f(mgda_optimize(x0 + s * v/ np.linalg.norm(v))) #corrector_step(x0 + s * v/ np.linalg.norm(v), L_vec_func = problem.f, G_mat_func = problem.grad, sigma = 0.01, lr=s,h_mode="exact", h_kwargs=None))) #f(mgda_optimize(x0 + s * v/ np.linalg.norm(v) ))
        print("corr", corr)
        ax_pf.plot([fi[0], corr[0]], [fi[1], corr[1]], [fi[2],corr[2]],"-o", c="blue",linewidth=2)
        if idx ==0:
            ax_pf.plot([fi[0], corr[0]], [fi[1], corr[1]], [fi[2],corr[2]],"-o", c="blue",label = "Corrector",linewidth=2)
       

ax_pf.set_xlabel(r"$𝒇_1(𝓍)$", fontsize = 20)
ax_pf.set_ylabel(r"$𝒇_2(𝓍)$", fontsize = 20)
ax_pf.set_zlabel(r"$𝒇_3(𝓍)$",labelpad=10, fontsize = 20)
ax_pf.tick_params(axis='x', labelsize=14)
ax_pf.tick_params(axis='y', labelsize=14)
ax_pf.tick_params(axis='z', labelsize=14)
ax_pf.view_init(elev=30, azim=13) #azim=60
ax_pf.legend(fontsize = 14)
ax_pf.set_title(r'$Objective \ space \ (\eta = 0.1)$', fontsize = 14)
ax_pf.grid(True)
plt.tight_layout()
plt.show()



f0 [0.34951935 0.21182564 0.64841309]
corr [0.34504125 0.15743412 0.71097901]
corr [0.33780438 0.1599869  0.71927121]
corr [0.32982775 0.16405017 0.72756044]
corr [0.32155922 0.16982012 0.73501958]
corr [0.31365997 0.17715367 0.74072205]
corr [0.30680311 0.18551039 0.74401185]
corr [0.30141856 0.19412954 0.74479716]
corr [0.29757939 0.202325   0.74351631]
corr [0.29508481 0.20967286 0.74084854]
[0.29254145 0.22633916 0.73058933]
[0.36307701 0.1537681  0.69120634]
[0.40417448 0.24626024 0.56393611]


In [20]:
# To see which directions are impossible or oscillating, we check the three conditions for all directions with label according to the preference vectors in Ps.
import math
%matplotlib tk

threshold = 1e-4
threshold2 = 1e-1
prev_v= None
vi = []
old_vi =[]
#for pref in range(2):
for pref in Ps:
    d = pref
    v_ud = compute_descent_direction(problem, x0, alpha, g_normalized, pref, solvers = "minres", reg =0.1) 
    #v_ud = compute_v_u_d(d)  # Compute v_{u_d}
    

    # Normalize both for visualization
    #d = d / np.linalg.norm(d)
    d_normalized = d / np.linalg.norm(d)

    d_y_proj = g0 @ v_ud     # or use: f(x0 + ε v_ud) - f(x0)

    # Normalize both for visualization
    d_y_proj = d_y_proj / np.linalg.norm(d_y_proj)

    alpha_normalized = alpha / np.linalg.norm(alpha)

    dot_product = np.dot(d_y_proj,d_normalized )
    max_alpha = np.linalg.norm(alpha_normalized, ord=np.inf)
    max_index = np.where(np.abs(alpha_normalized) == max_alpha)[0].item()
    max_2_index = np.where(np.abs(alpha_normalized) == np.sort(alpha_normalized)[-2])[0].item()


    #print("inf",max_alpha)

    # (a) Dot product threshold
    condition_a = dot_product <= threshold

    # (b) Sign oscillation
    condition_b = False
    """if i > 0 and prev_v is not None:
        sign_change = np.sign(d_y_proj) == -np.sign(prev_v)
        condition_b = np.all(sign_change)  # all coordinates flipped sign"""

    sign_change = np.sign(d_y_proj) == -np.sign(d)
    condition_b = np.all(sign_change)
    # (c) Alpha ∞-norm threshold
    eps_2 = 1-threshold2
    condition_c = max_alpha >= eps_2 

    if condition_c:
        print(f"At corner,Direction {d} exceeds alpha ∞-norm: {max_alpha}; alpha: {alpha_normalized}")
        if max_alpha == 1:
            print(f"At extreme point/minimum, take combination with {max_index + 1} to move away from corner")

        elif 1-threshold <= max_alpha < 1:
            print(f"Closer to an extreme point/minimum of objective {max_index + 1}, take combination with objective with {max_index + 1} to move closer or away")
            
        elif 1-threshold > max_alpha >= eps_2:
            print(f"At a corner of pareto set, avoid a combination of objective {max_index + 1} and {max_2_index+1}, and a combination with objective {max_index + 1} to strickly move away from the corner")
        else: 
            print(f"At extreme point/minimum")
            
        if condition_a:
            print(f"Direction {d} impossible -- (dot too small): {dot_product}")
        else: 
            vi.append((v_ud, 'tab:orange', f'{d}' , -s, s, 2))
        
    elif condition_a:
        print(f"Direction {d} impossible -- (dot too small): {dot_product}")
        vi.append((v_ud, 'tab:orange', f'{d}' , -s, s, 2))
    elif condition_b:
        print(f"Direction {d} oscillation -- sign flip: {d_y_proj}")
        vi.append((v_ud, 'tab:orange', f'{d}' , -s, s, 2))

    else:
        #vi.append((v_ud, 'tab:orange', f'MINRES{d}' , -s, s, 2))
        vi.append((v_ud, 'tab:orange', f'{d}' , -s, s, 2))
    #vi.append((v, 'tab:orange', 'MINRES' if i == 0 else None, -s, s, 2))
    #prev_v = d_y_proj 

#print(minres_solver(rhs))
#vi.append((minres_solver(rhs), 'tab:orange', 'MINRES', 0, s, 4))
#vi.append((minres_solver(rhs), 'tab:orange', 'MINRES' , -s, s, 2))
#vi.append((minres_solver(rhs), 'tab:orange', 'MINRES' if i == 0 else None, -s, s, 2))
#vi.append((v_ud, 'tab:orange', 'MINRES' , -s, s, 2))
vi.append((g0[0], 'tab:blue', r'$\nabla f_1(x^*)$', 0, s, 4))
vi.append((g0[1], 'tab:green', r'$\nabla f_2(x^*)$', 0, s, 4))
vi.append((g0[2], 'black', r'$\nabla f_3(x^*)$', 0, s, 4))

# Plot the Pareto set.
"""fig_ps = plt.figure(figsize=(10, 10))
ax_ps = fig_ps.add_subplot(111, projection='3d', proj_type='ortho')
problem.plot_pareto_set(ax_ps)
ax_ps.scatter(x0[1], x0[2], x0[0], c='tab:red', s=50)
for idx, (v, color, _, _, _, _) in enumerate(vi):
    draw_arrow_3d(ax_ps, x0 + v / np.linalg.norm(v), x0, color)
    #ax_ps.set_xlim([-2, 2])
    #ax_ps.set_ylim([-2, 2])

ax_ps.set_xlabel('x1')
ax_ps.set_ylabel('x2')
ax_ps.set_zlabel('x3')
ax_ps.legend()
plt.show()"""

points = vi
fig_ps = plt.figure(figsize=(10, 10))
ax_ps = fig_ps.add_subplot(111, projection='3d', proj_type='ortho')
#problem.plot_pareto_set(ax_ps)
ax_ps.scatter(x0[0], x0[1], x0[2], c='tab:red', s=50)
for idx, (v, color, label,_,_,_) in enumerate(points):
    #draw_arrow_3d(ax_ps, x0 + v / np.linalg.norm(v), x0, color)
    
    if idx == len(points)-1 or idx ==len(points)-2 or idx ==len(points)-3:
        new_x = x0 - s *v / np.linalg.norm(v)
        ax_ps.plot([x0[0], new_x[0]], [x0[1], new_x[1]], [x0[2],new_x[2]],"-", c=color,label = label,linewidth=2)
    else:
        new_x = x0 + s *v / np.linalg.norm(v)
        ax_ps.plot([x0[0], new_x[0]], [x0[1], new_x[1]], [x0[2],new_x[2]],"-", c=color,linewidth=2)
        if idx ==0:
            ax_ps.plot([x0[0], new_x[0]], [x0[1], new_x[1]], [x0[2],new_x[2]],"-", c=color,label = "directions",linewidth=2)
        corr= mgda_optimize(new_x)
        ax_ps.plot([new_x[0], corr[0]], [new_x[1], corr[1]], [new_x[2],corr[2]],"-o", c="blue",linewidth=2)
        if idx ==0:
            ax_ps.plot([new_x[0], corr[0]], [new_x[1], corr[1]], [new_x[2],corr[2]],"-o", c="blue",label = "corrector",linewidth=2)

        # This places the label closer to new_x, away from the potentially crowded origin.
        t_along_line = 0.8 # Adjust this value (0.0 to 1.0) to place the label along the line

        label_x = x0[0] + t_along_line * (new_x[0] - x0[0])
        label_y = x0[1] + t_along_line * (new_x[1] - x0[1])
        label_z = x0[2] + t_along_line * (new_x[2] - x0[2])

        # Add the text annotation
        ax_ps.text(label_x, label_y, label_z, label,
                color=color, # Use the line's color for the label
                fontsize=11,
                ha='center', # Horizontal alignment
                va='bottom', # Vertical alignment
                zdir=None,   # Text faces the viewer. For vectors, this is often good.
                            # Alternatively, zdir='z' might make text more upright.
                            # Experiment to see what looks best for your specific plot.
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.2'))

        
    #ax, head, tail,
    #[tail[1], head[1]], [tail[2], head[2]], [tail[0], head[0]]
    #ax_ps.set_xlim([-2, 2])
    #ax_ps.set_ylim([-2, 2])
"""ax_ps.set_xlabel('x1')
ax_ps.set_ylabel('x2')
ax_ps.set_zlabel('x3')
ax_ps.legend()
plt.show()"""

ax_ps.set_xlabel(r"$𝓍_1$",fontsize = 20)
ax_ps.set_ylabel(r'$𝓍_2$',fontsize = 20)
ax_ps.set_zlabel(r'$𝓍_3$', fontsize = 20)
ax_ps.set_title('Decision Space')
ax_ps.tick_params(axis='x', labelsize=14)
ax_ps.tick_params(axis='y', labelsize=14)
ax_ps.tick_params(axis='z', labelsize=14)
#ax_ps.view_init(elev=30, azim=60)
ax_ps.legend(fontsize = 14)
#plt.savefig("pareto_setxcalldxcqs13.png", dpi=300)
#ax_ps.grid(True)
plt.show()

# Plot the Pareto front.
fig_pf = plt.figure(figsize=(8, 8))
ax_pf = fig_pf.add_subplot(111, projection='3d')
print("f0 ", f0)
ax_pf.scatter(f0[0], f0[1],f0[2], c='tab:red', label='Pareto optimal $f(x^*)$', s=50)
#problem.plot_pareto_front(ax_pf, label='Pareto front')
"""for idx, (v, color, label, s0, s1, _) in enumerate(vi):
    fi = [problem.f(x0 + si * v / np.linalg.norm(v)) for si in np.linspace(s0, s1, 2)]
    fi = ndarray(fi)
    ax_pf.plot(fi[:, 0], fi[:, 1], c=color, label=label, linewidth=3)"""


       

for idx, (v, color, label, s0, s1, _)  in enumerate(vi):
    
    if idx == len(vi)-1 or idx ==len(vi)-2 or idx ==len(vi)-3:
        fi = problem.f(x0 - s * v / np.linalg.norm(v))
        #print(fi)
        ax_pf.plot([f0[0], fi[0]], [f0[1], fi[1]], [f0[2],fi[2]],"-", c=color,label = label,linewidth=2)

    else:
        fi = problem.f(x0 + s * v / np.linalg.norm(v))
        #print(fi, "h")
        ax_pf.plot([f0[0], fi[0]], [f0[1], fi[1]], [f0[2],fi[2]],"-", c=color,linewidth=2)
        if idx ==0:
            ax_pf.plot([f0[0], fi[0]], [f0[1], fi[1]], [f0[2],fi[2]],"-", c=color,label = "directions",linewidth=2)
        corr= problem.f(mgda_optimize(x0 + s * v / np.linalg.norm(v)))
        ax_pf.plot([fi[0], corr[0]], [fi[1], corr[1]], [fi[2],corr[2]],"-o", c="blue",linewidth=2)
        if idx ==0:
            ax_pf.plot([fi[0], corr[0]], [fi[1], corr[1]], [fi[2],corr[2]],"-o", c="blue",label = "Corrector",linewidth=2)

        
        t_along_line = 1.0 # Adjust this value (0.0 to 1.0) to place the label along the line

        label_x = f0[0] + t_along_line * (fi[0] - f0[0])
        label_y = f0[1] + t_along_line * (fi[1] - f0[1])
        label_z = f0[2] + t_along_line * (fi[2] - f0[2])

        # Add the text annotation
        ax_pf.text(label_x, label_y, label_z, label,
                color=color, # Use the line's color for the label
                fontsize=8,
                ha='center', # Horizontal alignment
                va='bottom', # Vertical alignment
                zdir=None,   # Text faces the viewer. For vectors, this is often good.
                            # Alternatively, zdir='z' might make text more upright.
                            # Experiment to see what looks best for your specific plot.
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.2'))
        



#print(fi)
"""ax_pf.set_xlabel('f1')
ax_pf.set_ylabel('f2')
ax_pf.set_zlabel('f3')
ax_pf.legend()
plt.show()"""
ax_pf.set_xlabel(r"$𝒇_1(𝓍)$", fontsize = 20)
ax_pf.set_ylabel(r"$𝒇_2(𝓍)$", fontsize = 20)
ax_pf.set_zlabel(r"$𝒇_3(𝓍)$",labelpad=10, fontsize = 20)
ax_pf.tick_params(axis='x', labelsize=14)
ax_pf.tick_params(axis='y', labelsize=14)
ax_pf.tick_params(axis='z', labelsize=14)
ax_pf.view_init(elev=30, azim=60)
ax_pf.legend(fontsize = 14)
ax_pf.set_title('Objective Space')
#plt.savefig("pareto_frontxcalldxcqs13.png", dpi=300)
ax_pf.grid(True)
plt.show()


#print("corr",problem.f(mgda_optimize(x0 + s * v_ud / np.linalg.norm(v_ud))))


f0  [0.2246733  0.21046102 0.97367784]


## For more than one step direction (All possible finite combinations)

In [7]:
import numpy as np

"""def generate_direction_vectors(num_per_pair=50):
    directions = []

    # Obj2 and Obj3 (ignore Obj1)
    for w in np.linspace(0, 1, num_per_pair):
        directions.append(np.array([0.0, -w, -(1 - w)]))

    # Obj1 and Obj3 (ignore Obj2)
    for w in np.linspace(0, 1, num_per_pair):
        directions.append(np.array([-w, 0.0, -(1 - w)]))

    # Obj1 and Obj2 (ignore Obj3)
    for w in np.linspace(0, 1, num_per_pair):
        directions.append(np.array([-w, -(1 - w), 0.0]))

    return directions"""

def generate_direction_vectors(num_per_pair=50):
    directions = []

    # Obj2 and Obj3 (ignore Obj1)
    for w in np.linspace(0, 1, num_per_pair):
        directions.append(np.array([0.0, -w, -(1 - w)]))

    # Obj1 and Obj3 (ignore Obj2)
    for w in np.linspace(0, 1, num_per_pair):
        directions.append(np.array([-w, 0.0, -(1 - w)]))

    # Obj1 and Obj2 (ignore Obj3)
    for w in np.linspace(0, 1, num_per_pair):
        directions.append(np.array([-w, -(1 - w), 0.0]))

    # Obj1
    for w in np.linspace(0.01, 1, num_per_pair):
        directions.append(np.array([-w, 0.0, 0.0]))
    # Obj2
    for w in np.linspace(0.01, 1, num_per_pair):
        directions.append(np.array([0.0, -w, 0.0]))

    # Obj3
    for w in np.linspace(0.01, 1, num_per_pair):
        directions.append(np.array([0.0, 0.0, -w]))
    return directions

# Usage


In [ ]:
%matplotlib tk
vi = []
threshold = 1e-6
threshold2 = 1e-1


dir =generate_direction_vectors(num_per_pair=50)

prev_v= None
for i, pref in enumerate(dir): 
    d_y = pref
    v_ud= compute_descent_direction(problem, x0, alpha, g0, pref)
    
    d_normalized = d_y / np.linalg.norm(d_y)

    d_y_proj = g0 @ v_ud     # or use: f(x0 + ε v_ud) - f(x0)

    # Normalize both for visualization
    d_y_proj = d_y_proj / np.linalg.norm(d_y_proj)

    alpha_normalized = alpha / np.linalg.norm(alpha)

    dot_product = np.dot(d_y_proj,d_normalized )
    max_alpha = np.linalg.norm(alpha_normalized, ord=np.inf)
    max_index = np.where(np.abs(alpha_normalized) == max_alpha)[0].item()
    max_2_index = np.where(np.abs(alpha_normalized) == np.sort(alpha_normalized)[-2])[0].item()


    #print("inf",max_alpha)

    # (a) Dot product threshold
    condition_a = dot_product <= threshold

    # (b) Sign oscillation
    condition_b = False
    """if i > 0 and prev_v is not None:
        sign_change = np.sign(d_y_proj) == -np.sign(prev_v)
        condition_b = np.all(sign_change)  # all coordinates flipped sign"""

    sign_change = np.sign(d_y_proj) == -np.sign(d_y)
    condition_b = np.all(sign_change)
    # (c) Alpha ∞-norm threshold
    eps_2 = 1-threshold2
    condition_c = max_alpha >= eps_2 

    if condition_c:
        print(f"At corner,Direction {d_y} exceeds alpha ∞-norm: {max_alpha}; alpha: {alpha_normalized}")
        if max_alpha == 1:
            print(f"At extreme point/minimum, take combination with {max_index + 1} to move away from corner")

        elif 1-threshold <= max_alpha < 1:
            print(f"Closer to an extreme point/minimum of objective {max_index + 1}, take combination with objective with {max_index + 1} to move closer or away")
            
        elif 1-threshold > max_alpha >= eps_2:
            print(f"At a corner of pareto set, avoid a combination of objective {max_index + 1} and {max_2_index+1}, and a combination with objective {max_index + 1} to strickly move away from the corner")
        else: 
            print(f"At extreme point/minimum")
    
        if condition_a:
            print(f"Direction {d_y} impossible -- (dot too small): {dot_product}")
        else: 
            vi.append((v_ud, 'tab:orange', 'predictor'if i == 0 else None  , -s, s, 2))
        
    elif condition_a:
        print(f"Direction {d_y} impossible -- (dot too small): {dot_product}")
    elif condition_b:
        print(f"Direction {d_y} oscillation -- sign flip: {d_y_proj}")

    else:
        vi.append((v_ud, 'tab:orange', 'predictor'if i == 0 else None , -s, s, 2))
        #vi.append((v, 'tab:orange', 'MINRES' if i == 0 else None, -s, s, 2))
        prev_v = d_y_proj 
vi.append((g0[0], 'tab:blue', r'$\nabla f_1(x^*)$', 0, s, 4))
vi.append((g0[1], 'tab:green', r'$\nabla f_2(x^*)$', 0, s, 4))
vi.append((g0[2], 'black', r'$\nabla f_3(x^*)$', 0, s, 4))



# Plot the Pareto set.
fig_ps = plt.figure(figsize=(8, 8))
ax_ps = fig_ps.add_subplot(111, projection='3d', proj_type='ortho')
#problem.plot_pareto_set(ax_ps)
ax_ps.scatter(x0[0], x0[1], x0[2], c='tab:red', s=100)
for idx, (v, color, label, _, _, _) in enumerate(vi):
    #draw_arrow_3d(ax_ps, x0 + v / np.linalg.norm(v), x0, color)
    
    if idx == len(vi)-1 or idx ==len(vi)-2 or idx ==len(vi)-3:
        new_x = x0 - s *v / np.linalg.norm(v)
        ax_ps.plot([x0[0], new_x[0]], [x0[1], new_x[1]], [x0[2],new_x[2]],"-", c=color,label = label,linewidth=2)
    else:
        new_x = x0 + s *v / np.linalg.norm(v)
        ax_ps.plot([x0[0], new_x[0]], [x0[1], new_x[1]], [x0[2],new_x[2]],"-", c=color,label = label,linewidth=2)
        corr= mgda_optimize(new_x) #corrector_step(new_x, L_vec_func = problem.f, G_mat_func = problem.grad, sigma = 0.01, lr=s,h_mode="exact", h_kwargs=None) #mgda_optimize(new_x)
        ax_ps.plot([new_x[0], corr[0]], [new_x[1], corr[1]], [new_x[2],corr[2]],"-", c="blue",linewidth=2)
        if idx ==0:
            ax_ps.plot([new_x[0], corr[0]], [new_x[1], corr[1]], [new_x[2],corr[2]],"-", c="blue",label = "Corrector",linewidth=2)

#ax_ps.set_xlabel(r"$𝓍_1$",fontsize = 20)
#ax_ps.set_ylabel(r'$𝓍_2$',fontsize = 20)
#ax_ps.set_zlabel(r'$𝓍_3$', fontsize = 20)
#ax_ps.set_title(r'$Decision \  space \ (\eta = 0.1)$', fontsize = 14)
ax_ps.tick_params(axis='x', labelsize=14)
ax_ps.tick_params(axis='y', labelsize=14)
ax_ps.tick_params(axis='z', labelsize=14)
ax_ps.view_init(elev=30, azim=60)
#ax_ps.legend()
#plt.savefig("pareto_setxcalldxcqs.png", dpi=300)
ax_ps.grid(False)
plt.show()



# Plot the Pareto front.
fig_pf = plt.figure(figsize=(8, 8))
ax_pf = fig_pf.add_subplot(111, projection='3d')
ax_pf.scatter(f0[0], f0[1],f0[2], c='tab:red', label='Pareto optimal $f(x^*)$', s=100)




#problem.plot_pareto_front(ax_pf, label='True front')
print("f0",f0)
for idx, (v, color, label, s0, s1, _) in enumerate(vi):
    
    if idx == len(vi)-1 or idx ==len(vi)-2 or idx ==len(vi)-3:
        fi = problem.f(x0 - s * v / np.linalg.norm(v))
        print(fi)
        ax_pf.plot([f0[0], fi[0]], [f0[1], fi[1]], [f0[2],fi[2]],"-", c=color,label = label,linewidth=2)

    else:
        fi = problem.f(x0 + s * v / np.linalg.norm(v))
        print(fi, "h")
        ax_pf.plot([f0[0], fi[0]], [f0[1], fi[1]], [f0[2],fi[2]],"-", c=color,label = label,linewidth=2)
        corr= problem.f(mgda_optimize(x0 + s * v/ np.linalg.norm(v) )) #corrector_step(x0 + s * v/ np.linalg.norm(v), L_vec_func = problem.f, G_mat_func = problem.grad, sigma = 0.01, lr=s,h_mode="exact", h_kwargs=None)) #(mgda_optimize(x0 + s * v / np.linalg.norm(v)))
        ax_pf.plot([fi[0], corr[0]], [fi[1], corr[1]], [fi[2],corr[2]],"-", c="blue",linewidth=2)
        if idx ==0:
            ax_pf.plot([fi[0], corr[0]], [fi[1], corr[1]], [fi[2],corr[2]],"-", c="blue",label = "Corrector",linewidth=2)

    
#ax_pf.legend()
#plt.show()
    
#ax_pf.set_xlabel(r"$𝒇_1(𝓍)$", fontsize = 20)
#ax_pf.set_ylabel(r"$𝒇_2(𝓍)$", fontsize = 20)
#ax_pf.set_zlabel(r"$𝒇_3(𝓍)$",labelpad=10, fontsize = 20)
ax_pf.tick_params(axis='x', labelsize=14)
ax_pf.tick_params(axis='y', labelsize=14)
ax_pf.tick_params(axis='z', labelsize=14)
ax_pf.view_init(elev=30, azim=60)
#ax_pf.legend()
#ax_pf.set_title(r'$Objective \ space \ (\eta = 0.1)$', fontsize = 14)
#plt.savefig("pareto_frontxcalldxcqs.png", dpi=300)
ax_pf.grid(False)
plt.show()



f0 [0.34951935 0.21182564 0.64841309]
[0.40441382 0.24578164 0.56402791] h
[0.41298735 0.21932229 0.57547937] h
[0.4130427  0.21855097 0.57602401] h
[0.41308941 0.21775629 0.57659626] h
[0.41312665 0.21693761 0.57719756] h
[0.4131535  0.21609429 0.5778294 ] h
[0.41316899 0.21522572 0.57849332] h
[0.41317208 0.21433131 0.57919095] h
[0.41316164 0.2134105  0.57992393] h
[0.41313652 0.21246279 0.58069399] h
[0.41309544 0.2114877  0.58150288] h
[0.41303709 0.21048483 0.58235239] h
[0.41296008 0.20945385 0.58324437] h
[0.41286293 0.20839451 0.58418065] h
[0.41274413 0.20730667 0.5851631 ] h
[0.41260206 0.20619027 0.58619358] h
[0.41243508 0.20504541 0.58727394] h
[0.41224148 0.20387233 0.58840598] h
[0.4120195  0.20267141 0.58959145] h
[0.41176737 0.20144322 0.59083204] h
[0.41148329 0.20018852 0.59212931] h
[0.41116544 0.19890828 0.5934847 ] h
[0.41081205 0.1976037  0.59489951] h
[0.41042135 0.19627621 0.59637482] h
[0.40999165 0.19492751 0.5979115 ] h
[0.40952133 0.19355953 0.59951017] h
